# <font color='blue'> Module 3 : Functions, Multiple Tables, and Sub-queries </font>

Built-in functions are pre-compiled routines provided by the database engine to manipulate, calculate, and format data on the fly. They fall into three primary categories:

1. **Aggregate Functions:** Operate on a multi-row column and return a single summary value.
2. **Scalar/Scalar Math Functions:** Operate on individual values per row.
3. **Date/Time Functions:** Extract, format, or manipulate date strings.

---

## 1. Aggregate Functions

These functions look down a vertical column of values across multiple rows, crush them together, and return a single descriptive statistic.



### Syntax
```sql
SELECT FUNCTION_NAME(column_name) FROM table_name;
```

### Key Functions:
* **`SUM(col)`**: Adds up all numeric values in a column.
* **`AVG(col)`**: Calculates the arithmetic mean.
* **`COUNT(col)`** or **`COUNT(*)`**: Counts rows. `COUNT(col)` skips missing (`NULL`) values; `COUNT(*)` counts every row regardless of content.
* **`MIN(col)`** / **`MAX(col)`**: Finds the smallest or largest value (works on numbers, dates, and text alphabetically).

---

## 2. Scalar & Conditional Functions

Scalar functions operate horizontally on individual values within a single row. If your query returns 100 rows, a scalar function runs 100 times.

### Key Functions:
* **`ROUND(value, decimals)`**: Rounds a float or decimal to a specified number of decimal places.
* **`IFNULL(value, replacement)`** *(SQLite specific)*: Evaluates an entry. If the entry is missing (`NULL`), it automatically swaps it out with your chosen replacement value. 
  > **Note:** In enterprise SQL, this is often written as `COALESCE(value, replacement)`.

### Syntax
```sql
SELECT ROUND(base_price * 0.10, 2) FROM cars;
SELECT IFNULL(commission_pct, 0) FROM sales;
```

## 3. Date and Time Functions

Because dates are often stored as text strings (`'YYYY-MM-DD'`), databases provide special functions to dissect them without messy string parsing.

In SQLite, the **`strftime(format, column)`** function is the Swiss Army knife for dates. It reads a date string and extracts specific components based on formatting tokens:

* **`%Y`** = Year (e.g., 2026)
* **`%m`** = Month as a number (01 through 12)
* **`%d`** = Day of the month (01 through 31)

### Syntax
```sql
SELECT strftime('%m', sale_date) AS sale_month FROM sales;
```

In [3]:
# Let's start with generating a new dataset for 'Car Sales'

import sqlite3
import pandas as pd

# Connect to a fresh in-memory database for this module
conn = sqlite3.connect(':memory:')

# 1. Create and populate the 'cars' table
conn.execute("""
CREATE TABLE cars (
    car_id INT PRIMARY KEY,
    make TEXT,
    model TEXT,
    body_style TEXT,
    base_price REAL
);
""")

conn.execute("""
INSERT INTO cars VALUES 
    (101, 'Toyota', 'Corolla', 'Sedan', 22000.00),
    (102, 'Honda', 'Civic', 'Sedan', 25000.00),
    (103, 'Ford', 'F-150', 'Truck', 45000.00),
    (104, 'Tesla', 'Model Y', 'SUV', 48000.00),
    (105, 'BMW', '3 Series', 'Sedan', 44000.00);
""")

# 2. Create and populate the 'sales' table
conn.execute("""
CREATE TABLE sales (
    sale_id INT PRIMARY KEY,
    car_id INT,
    sale_date DATE,
    sold_price REAL,
    commission_pct REAL
);
""")

conn.execute("""
INSERT INTO sales VALUES 
    (1, 101, '2026-01-15', 21500.00, 0.05),
    (2, 103, '2026-02-10', 46000.00, 0.04),
    (3, 104, '2026-02-28', 48000.00, NULL),  
    (4, 101, '2026-03-05', 22000.00, 0.05),
    (5, 102, '2026-03-12', 24500.00, 0.03);
""")                                           # Test case for built-in functions
conn.commit()

print("Setup Complete! 'cars' and 'sales' tables are ready for practice.")

Setup Complete! 'cars' and 'sales' tables are ready for practice.


## Aggregate functions

### `SUM`, `AVG`, `COUNT`, `MIN` and `MAX`
<font color='blue'>**Exercise 1: Inventory Value**</font>   
Calculate the total value and the average price of all vehicles currently sitting in the `cars` inventory.
- Expected Columns: Your query should return two summary columns. Let's alias them as `total_inventory_value` and `average_car_price`.
- Table to use: `cars`

In [8]:
import pandas as pd

df = pd.read_sql_query("""
SELECT 
    SUM(base_price) AS total_inventory_value,
    AVG(base_price) AS average_car_price
FROM cars;
""", conn)

print(df)

   total_inventory_value  average_car_price
0               184000.0            36800.0


<font color='blue'>**Exercise 2: Tricky Commission Counts**</font>   
Prove the difference between `COUNT(*)` (all values) and `COUNT(column_name)` (non-NULL values only). Write a single query that counts the total number of sales records versus the number of sales that actually recorded a commission percentage.

- Expected Columns: Alias them as `total_sales_records` and `sales_with_commission`.
- Table to use: `sales`

In [10]:
df2 = pd.read_sql_query("""
SELECT 
    COUNT(commission_pct) AS sales_with_commission,
    COUNT(*) AS total_sales_records
FROM sales;
""", conn)

print(df2)

   sales_with_commission  total_sales_records
0                      4                    5


<font color='blue'>**Exercise 3: Identifying the Extremes**</font>   
Find the absolute highest price a car actually sold for, and the lowest price a car actually sold for in the dealership's history.

- Expected Columns: Alias them as `highest_sale` and `lowest_sale`.
- Table to use: `sales`

In [13]:
df3 = pd.read_sql_query("""
SELECT
    MAX(sold_price) AS highest_sale,
    MIN(sold_price) AS lowest_sale
FROM sales;
""", conn)

print(df3)

   highest_sale  lowest_sale
0       48000.0      21500.0


## Scalar & Conditional Functions

### `ROUND` and `IFNULL`

<font color='blue'>**Exercise 4: Precision Calculations**</font>  
The dealership wants to see a 10% promotional discount calculated for every vehicle in stock, but the finance team needs the results rounded cleanly to 2 decimal places.  

- Expected Columns: make, model, base_price, and a new calculated column aliased as promo_discount.
- Table to use: cars

df4 = pd.read_sql_query("""
SELECT
    make,
    model,
    base_price,
    ROUND (base_price*0.1, 2) AS promo_discount
FROM cars;
""", conn)

print(df4)

<font color='blue'>**Exercise 5: Cleaning Up Missing Comissions**</font>  
Some sales entries have a `NULL` commission percentage because the salesperson didn't earn one (or it wasn't recorded). Generate a report showing the commission rates, but automatically replace any `NULL` values with `0.0`. 

- Expected Columns: `sale_id`, `sold_price`, and a new column aliased as `clean_commission`.
- Table to use: `sales`

In [27]:
df5 = pd.read_sql_query("""
SELECT
    sale_id,
    sold_price,
    IFNULL(commission_pct, 0.0) AS clean_commission
FROM sales;
""", conn)

print(df5)

   sale_id  sold_price  clean_commission
0        1     21500.0              0.05
1        2     46000.0              0.04
2        3     48000.0              0.00
3        4     22000.0              0.05
4        5     24500.0              0.03


<font color='blue'>**Exercise 6: Extracting the Sale Month**</font>  
The marketing team wants to analyze seasonal trends. Write a query that extracts just the month number from every sale date so they can see what time of year the transactions took place.
    
- Expected Columns: `sale_id`, `sale_date`, and a new column aliased as `sale_month`.
- Table to use: `sales`

In [ ]:
## Date & Time Functions

### `strftime` and `IFNULL`

In [31]:
df6 = pd.read_sql_query("""
SELECT
    sale_id,
    sale_date,
    strftime('%m', sale_date) AS sale_month
FROM sales;
""", conn)

print(df6)

   sale_id   sale_date sale_month
0        1  2026-01-15         01
1        2  2026-02-10         02
2        3  2026-02-28         02
3        4  2026-03-05         03
4        5  2026-03-12         03
